Notebook to plot 10m Temperature and weighted albedo output from the regularized albedo TOP experiments. This is to show that this parameterization does not produce unphysical results.

In [ ]:
from AOSCMcoupling import OIFSPreprocessor, NEMOPreprocessor
import xarray as xr
import proplot as pplt
import pandas as pd
from pathlib import Path
import warnings
import numpy as np

In [ ]:
dates = pd.date_range("2020-04-12 00:00", "2020-04-18 22:00", freq="2h")
tldir = Path("/home/valentina/dev/aoscm/experiments/output/top_ensemble_reg")
assert tldir.is_dir()

In [ ]:
def get_date_dirs(dates: pd.DatetimeIndex) -> list[Path]:
    date_dirs = [tldir / f"{date.date()}_{date.hour:02}" for date in dates]
    for date_dir in date_dirs:
        assert date_dir.is_dir()
    return date_dirs

In [ ]:
def load_single_iteration_results(
    dates: pd.DatetimeIndex, iter: int, file_name: str, preprocess: list[callable]
) -> xr.Dataset:
    date_dirs = get_date_dirs(dates)
    start_date_dim = xr.DataArray(dates, dims="start_date")
    date_files = []
    for date_dir, preproc in zip(date_dirs, preprocess):
        file = date_dir / f"parallel/{file_name}"
        ds = xr.open_mfdataset(str(file), preprocess=preproc)
        date_files.append(ds)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        concatenated_ds = xr.concat(date_files, start_date_dim)
    return concatenated_ds

In [ ]:
oifs_preproc = [OIFSPreprocessor(date).preprocess for date in dates]
progvar = load_single_iteration_results(dates, 1, "progvar.nc", oifs_preproc)

In [ ]:
nemo_preproc = [NEMOPreprocessor(date).preprocess for date in dates]
icemod = load_single_iteration_results(dates, 1, "*icemod*.nc", nemo_preproc)

In [ ]:
fig, axs = pplt.subplots(
    width="70em", height="40em", sharey=0, sharex=3, nrows=2
)

ax = axs[0]

t10 = progvar.t.isel(nlev=-1) - 273.15
for index in range(len(t10.start_date)):
    ax.plot(t10[index], color="k", alpha=0.5)

vmin, vmax = -35, 3
ax.format(
    xlabel="Time",
    ylabel="Temperature [°C]",
    xrotation=30,
    title="10m Temperature (Parallel Algorithm, Regularized Albedo)",
    ylim=[vmin, vmax],
)

ax = axs[1]

albedo = icemod.iceconc_cat.sel(ncatice=1) * icemod.icealb_cat.sel(ncatice=1)
for category in range(2, 6):
    albedo += icemod.iceconc_cat.sel(ncatice=category) * icemod.icealb_cat.sel(
        ncatice=category
    )

for index in range(len(albedo.start_date)):
    ax.plot(albedo[index], color="k", alpha=0.5)
vmin, vmax = 0.6, 0.9
ax.format(
    xlabel="Time",
    ylabel="Albedo [-]",
    xrotation=30,
    title="Sea Ice Albedo (Parallel Algorithm, Regularized Albedo)",
    ylim=[vmin, vmax],
)

axs.format(abc="a)")

fig.savefig("top_reg_parallel_output.png", dpi=300)